# Temporal Detection Evaluation

Evaluate cat and person tracks using temporal IoU, precision, recall and F1.

## Configure notebook

In [ ]:
from nb_utils import setup_notebook

PROJECT_ROOT = setup_notebook()

In [ ]:
from collections import defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.optimize import linear_sum_assignment

import batch_run
from config import settings
from nb_utils import (
    PERSON_LABEL_ID,
    VideoInfo,
    annotation_intervals,
    is_video_annotated,
    load_annotations,
)
from utils import get_video_paths

In [ ]:
# define constants
TIOU_THRESHOLD = 0.8

In [ ]:
# define paths
OUTPUT_DIR = PROJECT_ROOT / "object_clips"
METADATA_DIR = OUTPUT_DIR / "metadata"
LABELS_PATH = PROJECT_ROOT / "datasets" / "behaviour_dataset" / "behaviour_labels.json"

## Load target videos

In [ ]:
# all processed or annotated videos should be a subset of target videos
video_paths = get_video_paths(mock_inputs_long=True)
# video_paths = [Path("datasets/mock_inputs_long/20261026_082859.mp4")]
video_info = {p.name: VideoInfo(p) for p in video_paths}

## Load ground truth

In [ ]:
# identify labelled target videos and report unlabelled videos
annotations = load_annotations(LABELS_PATH)
ann_video_info = {
    n: v for n, v in video_info.items() if is_video_annotated(v, annotations)
}
unlabelled_videos = set(video_info) - set(ann_video_info)
print(f"Unlabelled videos: {len(unlabelled_videos)}")
for video_name in sorted(unlabelled_videos):
    print(f"  {video_name}")

# convert behaviour windows into class-specific presence intervals
windows = defaultdict(list)
for video_name, video in ann_video_info.items():
    intervals = annotation_intervals(annotations, video.hashes, video_name=video_name)
    for cat_id, start, end, _ in intervals:
        class_name = "person" if cat_id == PERSON_LABEL_ID else "cat"
        identity = cat_id if class_name == "cat" else "person"
        windows[(video_name, class_name, identity)].append((start, end))

# merge windows when the same object could remain in the track's stale state
ground_truth = []
for (name, class_name, _), intervals in windows.items():
    merged = []
    for start, end in sorted(intervals):
        gap = start - merged[-1][1] - 1 if merged else None
        if merged and gap < settings.TRACK_STALE_DUR * ann_video_info[name].fps:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    ground_truth.extend(
        {"video": name, "class": class_name, "start": start, "end": end}
        for start, end in merged
    )

# report the number of ground-truth intervals
print(f"Ground-truth intervals: {len(ground_truth)}")

## Run batch processing

In [ ]:
# # run the selected videos and surface batch failures
# return_code = batch_run.main(video_paths, clear_output=True)
# if return_code != 0:
#     raise RuntimeError(f"Batch processing failed with exit code {return_code}")

## Load predicted tracks

In [ ]:
# load exported tracks and report unprocessed target videos
tracks = [joblib.load(p) for p in METADATA_DIR.glob("track-*.joblib")]
proc_video_names = {
    n for t in tracks if (n := Path(t["video_path"]).name) in video_info
}
proc_video_info = {n: v for n, v in video_info.items() if n in proc_video_names}
unprocessed_videos = set(video_info) - set(proc_video_info)
print(f"Unprocessed videos: {len(unprocessed_videos)}")
for video_name in sorted(unprocessed_videos):
    print(f"  {video_name}")

# map prediction hashes to source-frame indices
video_hash_indices = {
    n: {h: i for i, h in enumerate(v.hashes)} for n, v in proc_video_info.items()
}

# convert exported tracks into source-frame intervals
predictions = []
for track in tracks:
    name = Path(track["video_path"]).name
    if name in proc_video_info:
        hash_indices = video_hash_indices[name]
        frame_indices = [
            hash_indices[h] for h in track["frame_hashes"] if h in hash_indices
        ]
        if frame_indices:
            predictions.append(
                {
                    "video": name,
                    "class": track["object_name"],
                    "start": min(frame_indices),
                    "end": max(frame_indices),
                }
            )
# report the number of predicted intervals
print(f"Predicted intervals: {len(predictions)}")

## Evaluate predictions

In [ ]:
def temporal_iou(a: dict, b: dict) -> float:
    """Calculate temporal intersection over union for inclusive frame intervals"""
    intersection = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]) + 1)
    union = (a["end"] - a["start"] + 1) + (b["end"] - b["start"] + 1) - intersection
    return intersection / union


# match tracks one-to-one within each video and class
matches = []
for video_name in ann_video_info:
    for class_name in ("cat", "person"):
        truth = [
            r
            for r in ground_truth
            if r["video"] == video_name and r["class"] == class_name
        ]
        tracks = [
            r
            for r in predictions
            if r["video"] == video_name and r["class"] == class_name
        ]
        if truth and tracks:
            overlaps = np.array([[temporal_iou(g, p) for p in tracks] for g in truth])
            rows, columns = linear_sum_assignment(
                np.where(overlaps >= TIOU_THRESHOLD, overlaps, 0), maximize=True
            )
            matches.extend(
                (truth[i], tracks[j])
                for i, j in zip(rows, columns)
                if overlaps[i, j] >= TIOU_THRESHOLD
            )

# store detection latency for each matched track
detection_latencies = pd.DataFrame(
    [
        {
            "video": g["video"],
            "class": p["class"],
            "latency_seconds": (p["start"] - g["start"]) / video_info[p["video"]].fps,
        }
        for g, p in matches
    ],
)

# count intervals and true positives per class and pooled
metrics = []
for class_name in ("cat", "person", "all"):
    truth = [r for r in ground_truth if class_name == "all" or r["class"] == class_name]
    tracks = [r for r in predictions if class_name == "all" or r["class"] == class_name]
    true_positives = sum(
        class_name == "all" or g["class"] == class_name for g, _ in matches
    )
    metrics.append(
        {
            "class": class_name,
            "ground-truth intervals": len(truth),
            "predicted tracks": len(tracks),
            "TP": true_positives,
        }
    )

# derive false positives, false negatives, precision, recall and F1
metrics = pd.DataFrame(metrics).set_index("class")
metrics["FP"] = metrics["predicted tracks"] - metrics["TP"]
metrics["FN"] = metrics["ground-truth intervals"] - metrics["TP"]
metrics["precision"] = metrics["TP"].div(metrics["predicted tracks"]).fillna(0)
metrics["recall"] = metrics["TP"].div(metrics["ground-truth intervals"]).fillna(0)
metrics["F1"] = (
    (2 * metrics["TP"]).div(2 * metrics["TP"] + metrics["FP"] + metrics["FN"]).fillna(0)
)

# show the summary table
metrics.round(3)

## Plot per-video timelines

In [ ]:
# find videos where intervals and matches disagree
video_names_with_errors = []
for video_name in ann_video_info:
    truth_count = sum(r["video"] == video_name for r in ground_truth)
    track_count = sum(r["video"] == video_name for r in predictions)
    match_count = sum(g["video"] == video_name for g, _ in matches)
    if not truth_count == track_count == match_count:
        video_names_with_errors.append(video_name)

# compare ground-truth and predicted intervals for videos with errors
if video_names_with_errors:
    fig, axes = plt.subplots(
        len(video_names_with_errors),
        1,
        figsize=(14, 2.5 * len(video_names_with_errors)),
        squeeze=False,
    )
    for ind, (axis, video_name) in enumerate(zip(axes[:, 0], video_names_with_errors)):
        for lane, class_name, intervals, color in (
            (0, "cat", ground_truth, "tab:green"),
            (1, "cat", predictions, "tab:blue"),
            (2, "person", ground_truth, "tab:orange"),
            (3, "person", predictions, "tab:red"),
        ):
            lane_intervals = [
                i
                for i in intervals
                if i["video"] == video_name and i["class"] == class_name
            ]
            for i, interval in enumerate(lane_intervals):
                overlapping_before = sum(
                    interval["start"] <= other["end"]
                    and other["start"] <= interval["end"]
                    for other in lane_intervals[:i]
                )
                overlaps_later = any(
                    interval["start"] <= other["end"]
                    and other["start"] <= interval["end"]
                    for other in lane_intervals[i + 1 :]
                )
                if overlapping_before or overlaps_later:
                    offset = 0.15 if overlapping_before % 2 else -0.15
                else:
                    offset = 0
                axis.plot(
                    [interval["start"], interval["end"]],
                    [lane + offset, lane + offset],
                    color=color,
                    linewidth=5,
                )
        axis.set_yticks(
            range(4), ["cat GT", "cat tracks", "person GT", "person tracks"]
        )
        axis.set_title(f"{ind}:   {video_name}")
        axis.set_xlabel("Source frame")
        axis.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    print("No videos with errors to plot")

## Plot detection latency

In [ ]:
# compare matched detection latency between object classes
fig, ax = plt.subplots(figsize=(15, 4))
sns.histplot(
    data=detection_latencies,
    x="latency_seconds",
    hue="class",
    stat="proportion",
    binwidth=0.2,
    common_norm=False,
    multiple="layer",
    element="step",
    fill=False,
    ax=ax,
)
ax.set_xlabel("Detection latency [s]")
ax.set_ylabel("Count")
ax.grid(axis="y")
fig.tight_layout()